# NSFW Moderation Head — `google/siglip2-base-patch16-naflex`

Trains a lightweight binary classifier (safe vs NSFW) on top of frozen naflex embeddings.

**Dataset:** `deepghs/nsfw_detect` (~28k images)  
**Output:** `nsfw_head.onnx` — input: float32[B, 768] normalized embedding → output: float32[B, 1] logit

**Runtime:** GPU (T4 recommended). Embedding step takes ~15 min; training is <1 min.

---
**Steps:**
1. Install deps & check GPU
2. Mount Drive (save embeddings + ONNX across sessions)
3. Load frozen naflex backbone
4. Load dataset & pre-compute embeddings
5. Train MLP head
6. Evaluate (AUC, accuracy)
7. Export to ONNX & verify

## 1. Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU — embedding step will be very slow")

In [ ]:
!pip install -q transformers datasets scikit-learn onnx onnxruntime pillow tqdm

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/currents_moderation"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Save directory:", SAVE_DIR)

## 3. Load Frozen Backbone

In [ ]:
from transformers import AutoModel, AutoProcessor
import torch
import torch.nn.functional as F

CHECKPOINT = "google/siglip2-base-patch16-naflex"
MAX_PATCHES = 256  # must match inference/main.py

processor = AutoProcessor.from_pretrained(CHECKPOINT)
backbone = AutoModel.from_pretrained(CHECKPOINT, torch_dtype=torch.float32)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False

device = "cuda" if torch.cuda.is_available() else "cpu"
backbone = backbone.to(device)
print(f"Backbone loaded on {device}")

In [ ]:
from tqdm.auto import tqdm
from PIL import Image

def embed_dataset_chunked(dataset, save_path, image_col="image", label_col="binary_label",
                          batch_size=32, save_every=4000):
    """Stream-embed an HF Dataset with periodic checkpointing.

    Streams batches so only ~batch_size PIL images live in RAM at once (avoids OOM).
    Writes progress to `save_path + '.partial'` every ~save_every samples; a later
    call resumes from the partial file if a crash/disconnect occurred.
    On completion writes the final pair to `save_path` and removes the partial.
    """
    if os.path.exists(save_path):
        print(f"Loading cached embeddings from {save_path}")
        ckpt = torch.load(save_path)
        return ckpt["embeddings"], ckpt["labels"]

    partial_path = save_path + ".partial"
    all_embs, all_lbls = [], []
    start_idx = 0
    if os.path.exists(partial_path):
        ckpt = torch.load(partial_path)
        all_embs = [ckpt["embeddings"]]
        all_lbls = [ckpt["labels"]]
        start_idx = ckpt["next_idx"]
        print(f"Resuming from {start_idx}/{len(dataset)}")

    pbar = tqdm(total=len(dataset), initial=start_idx, desc="Embedding")
    last_save = start_idx
    for i in range(start_idx, len(dataset), batch_size):
        rows = dataset[i:i + batch_size]
        images = [img.convert("RGB") if img.mode != "RGB" else img for img in rows[image_col]]
        inputs = processor(images=images, max_num_patches=MAX_PATCHES, return_tensors="pt").to(device)
        with torch.no_grad():
            feats = backbone.get_image_features(**inputs)
            if hasattr(feats, "pooler_output"):
                feats = feats.pooler_output
        feats = F.normalize(feats.cpu().float(), dim=-1)
        all_embs.append(feats)
        all_lbls.append(torch.tensor(rows[label_col], dtype=torch.float32))
        pbar.update(len(images))
        del rows, images, inputs, feats

        next_idx = i + batch_size
        if next_idx - last_save >= save_every:
            torch.save({
                "embeddings": torch.cat(all_embs),
                "labels": torch.cat(all_lbls),
                "next_idx": next_idx,
            }, partial_path)
            last_save = next_idx
    pbar.close()

    embeddings = torch.cat(all_embs)
    labels = torch.cat(all_lbls)
    torch.save({"embeddings": embeddings, "labels": labels}, save_path)
    if os.path.exists(partial_path):
        os.remove(partial_path)
    return embeddings, labels

## 4. Load Dataset & Pre-compute Embeddings

The dataset requires a Hugging Face account (NSFW content gate). Log in with your token if prompted.

In [ ]:
# Log in to Hugging Face (needed for gated dataset)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from datasets import load_dataset

ds = load_dataset("deepghs/nsfw_detect")
print(ds)

# Inspect label structure
split = list(ds.keys())[0]
print("\nFeatures:", ds[split].features)
print("Label classes:", ds[split].features["label"].names if hasattr(ds[split].features.get("label", None), "names") else "(inspect manually)")

In [ ]:
import numpy as np

# Classes: ['drawings', 'hentai', 'neutral', 'porn', 'sexy']
# 0 = safe (neutral, drawings), 1 = NSFW (hentai, porn, sexy)
SAFE_CLASSES = {"neutral", "drawings"}

label_names = ds["train"].features["label"].names

ds = ds.map(lambda ex: {"binary_label": 0 if label_names[ex["label"]] in SAFE_CLASSES else 1})

from datasets import concatenate_datasets
full = concatenate_datasets([ds[s] for s in ds.keys()])
full = full.shuffle(seed=42)
print(f"Total samples: {len(full)}")
print("Class balance (safe / nsfw):", np.bincount([ex["binary_label"] for ex in full]))

In [ ]:
EMBED_CACHE = f"{SAVE_DIR}/embeddings_nsfw.pt"
embeddings, labels = embed_dataset_chunked(full, EMBED_CACHE)
print(f"Embeddings shape: {embeddings.shape}, Labels shape: {labels.shape}")

## 5. Train MLP Head

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split
import torch.nn as nn

# 80/20 train/val split
dataset = TensorDataset(embeddings, labels)
n_val = int(len(dataset) * 0.2)
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=1024, shuffle=False)

print(f"Train: {n_train}, Val: {n_val}")

In [ ]:
class ModerationHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.net(x)


head = ModerationHead().to(device)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=0.01)
criterion = nn.BCEWithLogitsLoss()

EPOCHS = 20
best_val_loss = float("inf")
best_state = None

for epoch in range(1, EPOCHS + 1):
    # Train
    head.train()
    train_loss = 0.0
    for emb, lbl in train_loader:
        emb, lbl = emb.to(device), lbl.to(device)
        optimizer.zero_grad()
        loss = criterion(head(emb).squeeze(1), lbl)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(emb)
    train_loss /= n_train

    # Val
    head.eval()
    val_loss = 0.0
    with torch.no_grad():
        for emb, lbl in val_loader:
            emb, lbl = emb.to(device), lbl.to(device)
            val_loss += criterion(head(emb).squeeze(1), lbl).item() * len(emb)
    val_loss /= n_val

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in head.state_dict().items()}

    print(f"Epoch {epoch:02d}/{EPOCHS}  train={train_loss:.4f}  val={val_loss:.4f}")

head.load_state_dict(best_state)
print(f"\nBest val loss: {best_val_loss:.4f}")

## 6. Evaluate

In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

head.eval()
all_logits, all_labels = [], []
with torch.no_grad():
    for emb, lbl in val_loader:
        logits = head(emb.to(device)).squeeze(1).cpu()
        all_logits.append(logits)
        all_labels.append(lbl)

all_logits = torch.cat(all_logits).numpy()
all_labels = torch.cat(all_labels).numpy()
all_probs  = 1 / (1 + np.exp(-all_logits))
all_preds  = (all_probs >= 0.5).astype(int)

auc = roc_auc_score(all_labels, all_probs)
acc = accuracy_score(all_labels, all_preds)
print(f"AUC-ROC:  {auc:.4f}")
print(f"Accuracy: {acc:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=["safe", "nsfw"]))

## 7. Export to ONNX & Verify

In [ ]:
import onnx
import onnxruntime as ort

ONNX_PATH = f"{SAVE_DIR}/nsfw_head.onnx"

head_cpu = head.cpu()
head_cpu.eval()
dummy_input = torch.randn(1, 768)

torch.onnx.export(
    head_cpu,
    dummy_input,
    ONNX_PATH,
    input_names=["embedding"],
    output_names=["logits"],
    dynamic_axes={"embedding": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

onnx.checker.check_model(ONNX_PATH)
print(f"ONNX model saved and validated: {ONNX_PATH}")

In [ ]:
# Verify ONNX output matches PyTorch within tolerance
session = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])

test_emb = embeddings[:16].numpy()
with torch.no_grad():
    pt_logits = head_cpu(torch.from_numpy(test_emb)).squeeze(1).numpy()
ort_logits = session.run(None, {"embedding": test_emb})[0].squeeze(1)

max_diff = np.abs(pt_logits - ort_logits).max()
print(f"Max PyTorch vs ONNX logit difference: {max_diff:.2e}")
assert max_diff < 1e-4, f"ONNX mismatch too large: {max_diff}"
print("ONNX output verified ✓")
print(f"\nModel ready at: {ONNX_PATH}")